In [2]:
import torch
from ultralytics import YOLO
from torchvision import models
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import os


In [3]:
# --- Config ---
yolo_model_path = "YOLO8m-Experiments/left_right_signs_train/weights/best.pt"
cnn_model_path = "resnet18_left_right_best.pth"
image_dir = "D:/Desktop/test/road_signs_detection/Data2/images/test"
output_dir = "classified_output/"
img_size = 224
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# --- Load YOLO model ---
yolo = YOLO(yolo_model_path)
yolo_classes = yolo.model.names  # dict like {0: 'stop', 1: 'right_left'}

In [5]:
# --- Load CNN model ---
cnn = models.resnet18(pretrained=False)
cnn.fc = torch.nn.Linear(cnn.fc.in_features, 2)
cnn.load_state_dict(torch.load(cnn_model_path, map_location=device))
cnn = cnn.to(device)
cnn.eval()
cnn_classes = ['left', 'right']

# --- Preprocess for CNN ---
def preprocess_cnn(img_crop):
    img = img_crop.resize((img_size, img_size))
    img = np.array(img) / 255.0
    img = torch.tensor(img, dtype=torch.float).permute(2, 0, 1).unsqueeze(0).to(device)
    return img

d:\anaconda3\envs\ML_pro\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\anaconda3\envs\ML_pro\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [9]:
# --- Inference on Images ---
for filename in os.listdir(image_dir):
    if not filename.lower().endswith((".jpg", ".png")):
        continue

    img_path = os.path.join(image_dir, filename)
    image = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    # YOLO detection
    results = yolo(image)

    for box in results[0].boxes:
        class_id = int(box.cls)
        conf = float(box.conf)
        label = yolo_classes[class_id]
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        if label == "stop":
            final_label = f"stop ({conf:.2f})"
        elif label == "right_right":
            # Crop and classify
            crop = image.crop((x1, y1, x2, y2))
            input_tensor = preprocess_cnn(crop)
            with torch.no_grad():
                output = cnn(input_tensor)
                pred_class = cnn_classes[torch.argmax(output).item()]
            final_label = f"{pred_class} ({conf:.2f})"
        else:
            final_label = f"not stop left or right"

        # Draw box + label
        draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
        draw.text((x1, y1 - 10), final_label, fill="black")

    # Save result image
    output_path = os.path.join(output_dir, filename)
    image.save(output_path)
    print(f"✅ Saved: {output_path}")
    



0: 800x800 1 stop, 215.0ms
Speed: 6.0ms preprocess, 215.0ms inference, 1.5ms postprocess per image at shape (1, 3, 800, 800)
✅ Saved: classified_output/00014_00000_00003_png.rf.f969eacc1400bfc8246c74de2adafad7.jpg

0: 800x800 1 stop, 122.5ms
Speed: 4.6ms preprocess, 122.5ms inference, 1.8ms postprocess per image at shape (1, 3, 800, 800)
✅ Saved: classified_output/00014_00000_00022_png.rf.d67977226ad21c52aa98b52be01d3875.jpg

0: 800x800 1 stop, 22.7ms
Speed: 3.9ms preprocess, 22.7ms inference, 1.4ms postprocess per image at shape (1, 3, 800, 800)
✅ Saved: classified_output/00014_00000_00024_png_jpg.rf.7afe9e3e696eccc3357289b190d6a58b.jpg

0: 800x800 1 stop, 22.6ms
Speed: 4.0ms preprocess, 22.6ms inference, 1.7ms postprocess per image at shape (1, 3, 800, 800)
✅ Saved: classified_output/00014_00000_00025_png_jpg.rf.3a4c0c2e7663b5afac424ea9d8d42df4.jpg

0: 800x800 1 stop, 22.1ms
Speed: 3.3ms preprocess, 22.1ms inference, 1.4ms postprocess per image at shape (1, 3, 800, 800)
✅ Saved: cla